# Handoff

In [5]:
import os
import yfinance as yf
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool, handoff
from IPython.display import display, Markdown

load_dotenv()

True

In [2]:
@function_tool
def get_stock_data(ticker: str)->dict:
    """
    Fetches real time stock data from yahoo finance
    Returns price, P/E ratio, market cap , revenue growth, and profit margin for the given ticker

    Args:
        ticker: Stock ticker symbol (e.g., AAPL for Apple Inc.)
    """
    stock = yf.Ticker(ticker)
    info = stock.info

    data = {
        "ticker": ticker,
        "current_price": info.get("currentPrice"),
        "pe_ratio": info.get("trailingPE"),
        "market_cap": info.get("marketCap"),
        "52_week_high": info.get("fiftyTwoWeekHigh"),
        "52_week_low": info.get("fiftyTwoWeekLow"),
        "revenue_growth": info.get("revenueGrowth"),
        "profit_margin": info.get("profitMargins")
    }

    return data

In [ ]:
research_agent = Agent(
    name="ResearchAgent",
    model="gpt-4o-mini",
    instructions="""
    You are a specilaized financial research analyst.
    Always fetch real stock data before making claims.
    Provide structured analysis with:
    - key metrics
    - Bull case
    - Bear case
    - Overall assessment
    """,
    tools=[get_stock_data]
)

In [4]:
triage_agent = Agent(
    name="TriageAgent",
    model="gpt-4o-mini",
    instructions="""
    You are a gatekeeper for a financial research platform.

    Your only job:
    - If the user is asking about stocks, inverstments, or financial analysis => handoff to ResearchAgent
    - If the user is asking something unrelated => politely decline and explain this plateform is for inversment research only

    Do NOT answer research questions yourself. Always handoff.
    """,
    handoffs = [research_agent]
) 

In [7]:
resust = await Runner.run(
    triage_agent,
    "Should I invest in Apple? Give me a full analysis."
)
display(Markdown(f"**Final Result:** {resust.final_output}"))

**Final Result:** Investing in Apple (AAPL) can be a compelling opportunity, but it's essential to assess various factors before making a decision. Here’s a comprehensive analysis covering several key aspects: 

### 1. **Company Overview**
   - **Business Model & Products:** Apple Inc. is a technology company known for its innovative products, including the iPhone, iPad, Mac computers, Apple Watch, and services such as Apple Music, Apple TV+, and the App Store.
   - **Market Position:** Apple is one of the largest companies in the world by market capitalization, consistently competing with other tech giants like Microsoft, Amazon, and Google.

### 2. **Financial Performance**
   - **Revenue Growth:** Examine the revenue growth trends. Apple has a strong track record of increasing revenue, particularly due to its services segment, which has higher margins.
   - **Profit Margins:** Review gross and net profit margins. Apple typically maintains high profit margins compared to its peers.
   - **Cash Flow:** Strong cash flow generation allows Apple to reinvest in innovation, return capital to shareholders through dividends and buybacks, and maintain a significant cash reserve.

### 3. **Valuation Metrics**
   - **Price-to-Earnings (P/E) Ratio:** Compare Apple's P/E ratio to industry averages to determine if the stock is overvalued or undervalued.
   - **Price-to-Sales (P/S) Ratio:** Another important metric, especially given Apple's high revenue per share.
   - **Dividend Yield:** Look at the dividend yield; Apple has been increasing its dividends steadily, which can be appealing for income-focused investors.

### 4. **Growth Drivers**
   - **Product Innovation:** Apple's continuous pipeline of new products and technology (like AR/VR and potential automotive developments) can drive future growth.
   - **Ecosystem Expansion:** The integration of hardware, software, and services promotes customer loyalty and recurring revenue.
   - **Global Expansion:** Emerging markets represent significant growth opportunities.

### 5. **Risks**
   - **Market Saturation:** The smartphone market is approaching saturation, particularly in developed countries.
   - **Supply Chain Issues:** The dependence on global supply chains can present risks, as seen during the COVID-19 pandemic and geopolitical tensions.
   - **Regulatory Risks:** Increased scrutiny from governments regarding antitrust issues and privacy concerns can impact operations.

### 6. **Technical Analysis**
   - Review recent stock trends, moving averages, and support/resistance levels to gain insights into market sentiment.

### 7. **Analyst Opinions**
   - Review recommendations from financial analysts regarding Apple's stock—consider consensus ratings, target prices, and revisions.

### Conclusion:
Investing in Apple could be attractive due to its strong brand, innovative capacity, robust financials, and growing services segment. However, it's essential to weigh these factors against potential risks and your investment strategy. Assess your risk tolerance, time horizon, and whether this investment aligns with your overall portfolio goals.

### Recommendation:
Consider consulting with a financial advisor to tailor the investment decision to your personal financial circumstances and goals.

In [8]:
result2 = await Runner.run(
    triage_agent,
    "What's the best recipe for pasta carbonara?"
)
print(result2.final_output)

I'm here to assist with investment research and financial queries. For recipes, I recommend checking out a cooking website or resource. If you have any questions about stocks or investments, feel free to ask!


In [9]:
research_agent_v2 = Agent(
    name="ResearchAgent",
    model="gpt-4o-mini",
    instructions="""
    You are a specialist financial research analyst.
    
    STRICT RULE: You MUST call get_stock_data before writing anything.
    If you write any analysis without first calling the tool, you have failed.
    
    Your output structure:
    - Data Retrieved: [show actual numbers from tool]
    - Bull Case: 3 specific reasons backed by the data
    - Bear Case: 3 specific risks backed by the data  
    - Verdict: Buy / Hold / Avoid with one-sentence reasoning
    """,
    tools=[get_stock_data]
)

triage_agent_v2 = Agent(
    name="TriageAgent",
    model="gpt-4o-mini",
    instructions="""
    You are a gatekeeper for a financial research platform.
    
    - Investment/stock questions → hand off to ResearchAgent
    - Everything else → decline politely
    
    Do NOT answer research questions yourself.
    """,
    handoffs=[research_agent_v2]
)

In [11]:
result3 = await Runner.run(
    triage_agent_v2,
    "Should I invest in Apple? Give me a full analysis."
)
display(Markdown(f"**Final Result:** {result3.final_output}"))

**Final Result:** - **Data Retrieved:**
  - Current Price: $313.055
  - P/E Ratio: 37.86
  - Market Cap: $4.60 Trillion
  - 52-Week High: $316.94
  - 52-Week Low: $195.07
  - Revenue Growth: 16.6%
  - Profit Margin: 27.15%

- **Bull Case:**
  1. **Strong Revenue Growth:** Apple has a revenue growth rate of 16.6%, indicating robust demand for its products and services, which positions it well for future profitability.
  2. **High Profit Margin:** With a profit margin of 27.15%, Apple effectively converts a significant portion of its revenue into profit, suggesting operational efficiency and strong pricing power.
  3. **Market Leader:** Apple’s market capitalization of $4.60 trillion signifies its dominance in the tech industry, with a strong brand and loyal customer base, which could drive sustained growth.

- **Bear Case:**
  1. **High Valuation:** The P/E ratio of 37.86 is relatively high, suggesting that the stock may be overvalued compared to industry peers, leaving little room for error if earnings don't meet expectations.
  2. **Economic Sensitivity:** As a consumer electronics company, Apple’s sales could be vulnerable to economic downturns, which can adversely affect consumer spending patterns.
  3. **Increased Competition:** The tech market is highly competitive, with rivals continuously innovating. This competitive pressure could impact Apple's market share and profit margins over time.

- **Verdict:** **Hold** - While Apple exhibits strong growth potential and market leadership, the high valuation and competitive risks warrant a cautious approach before committing additional funds.